In [1]:
import os
import gc
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel

In [2]:
dataset_path = '/kaggle/input/datasets/fati22/tokopedia-images-product'
csv_path = "/kaggle/input/datasets/fati22/tokopedia-products-with-images-dataset-v1/df_final_clean.csv"

try:
    total_gambar = len(os.listdir(dataset_path))
    print(f"Total gambar yang terdeteksi: {total_gambar:,}")
except FileNotFoundError:
    print("Path tidak ditemukan, pastikan penulisan path sudah benar.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

existing_images = {f.replace('.jpg', '') for f in os.listdir(dataset_path) if f.endswith('.jpg')}
print(f"Total ID Product yang ditemukan dalam folder: {len(existing_images):,}")

Total gambar yang terdeteksi: 3,475,088
Total ID Product yang ditemukan dalam folder: 3,475,088


In [3]:
df = pd.read_csv(csv_path)
df_final = df[df["ID_Product"].astype(str).isin(existing_images)].copy()
print(f"Shape awal df_final: {df_final.shape}")

Shape awal df_final: (3475088, 6)


In [4]:
duplikat = df[df.duplicated(subset=['ID_Product'], keep=False)]
print(f"Jumlah baris duplikat: {len(duplikat)}")

Jumlah baris duplikat: 0


In [5]:
df_final = df_final.drop_duplicates(subset=['ID_Product'], keep='first')
print(f"Shape setelah drop duplikat: {df_final.shape}")

Shape setelah drop duplikat: (3475088, 6)


In [6]:
class TokopediaDataset(Dataset):
    def __init__(self, dataframe, img_dir, processor):
        self.df = dataframe
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        id_product = row['ID_Product']
        judul = row['Product_Name']
        img_name = f"{int(id_product)}.jpg"
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt")
            pixel_values = inputs['pixel_values'].squeeze(0)
        except Exception:
            pixel_values = torch.zeros(3, 224, 224)
        return pixel_values, str(id_product), str(judul)

In [7]:
model_name = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).cuda()
model.eval()

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/910 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: laion/CLIP-ViT-H-14-laion2B-s32B-b79K
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 1024)
      (position_embedding): Embedding(77, 1024)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-23): 24 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1024,), e

In [8]:
BATCH_SIZE = 32
chunk_split = [
    #(0, 250000, 1),
    #(250000, 500000, 2),
    #(500000, 750000, 3),
    (750000, 1000000, 4),
    #(1000000, 1250000, 5),
    #(1250000, 1500000, 6),
    #(1500000, 1750000, 7),
    #(1750000, 2000000, 8),
    #(2000000, 2250000, 9),
    #(2250000, 2500000, 10),
    #(2500000, 2750000, 11),
    #(2750000, 3000000, 12),
    #(3000000, None, 13)
]

In [9]:
for start, end, chunk_num in chunk_split:
    if end:
        df_chunk = df_final.iloc[start:end].copy()
    else:
        df_chunk = df_final.iloc[start:].copy()
    dataset = TokopediaDataset(df_chunk, dataset_path, processor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    all_img_vectors = []
    all_ids, all_juduls = [], []
    print(f"Memulai Ekstraksi Chunk {chunk_num}...")
    with torch.no_grad():
        for i, (imgs, ids, juduls) in enumerate(dataloader):
            imgs = imgs.cuda()
            vision_outputs  = model.vision_model(pixel_values=imgs)
            image_features  = model.visual_projection(vision_outputs.pooler_output)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            all_img_vectors.append(image_features.cpu().numpy())
            all_ids.extend(ids)
            all_juduls.extend(juduls)
            if i % 200 == 0:
                print(f"Chunk {chunk_num} -> Batch {i} selesai", flush=True)
    df_output = pd.DataFrame({
        'ID_Product'     : all_ids,
        'Judul'          : all_juduls,
        'Image_Embedding': np.vstack(all_img_vectors).tolist(),
    })
    
    nama_file = f"chunk_{chunk_num}.parquet"
    df_output.to_parquet(nama_file, engine='pyarrow')
    print(f"Chunk {chunk_num} selesai disave sebagai {nama_file}")
    del df_chunk, dataset, dataloader, df_output, all_img_vectors, all_ids, all_juduls
    gc.collect()
    torch.cuda.empty_cache()
print("Done!")

Memulai Ekstraksi Chunk 4...
Chunk 4 -> Batch 0 selesai
Chunk 4 -> Batch 200 selesai
Chunk 4 -> Batch 400 selesai
Chunk 4 -> Batch 600 selesai
Chunk 4 -> Batch 800 selesai
Chunk 4 -> Batch 1000 selesai
Chunk 4 -> Batch 1200 selesai
Chunk 4 -> Batch 1400 selesai
Chunk 4 -> Batch 1600 selesai
Chunk 4 -> Batch 1800 selesai
Chunk 4 -> Batch 2000 selesai
Chunk 4 -> Batch 2200 selesai
Chunk 4 -> Batch 2400 selesai
Chunk 4 -> Batch 2600 selesai
Chunk 4 -> Batch 2800 selesai
Chunk 4 -> Batch 3000 selesai
Chunk 4 -> Batch 3200 selesai
Chunk 4 -> Batch 3400 selesai
Chunk 4 -> Batch 3600 selesai
Chunk 4 -> Batch 3800 selesai
Chunk 4 -> Batch 4000 selesai
Chunk 4 -> Batch 4200 selesai
Chunk 4 -> Batch 4400 selesai
Chunk 4 -> Batch 4600 selesai
Chunk 4 -> Batch 4800 selesai
Chunk 4 -> Batch 5000 selesai
Chunk 4 -> Batch 5200 selesai
Chunk 4 -> Batch 5400 selesai
Chunk 4 -> Batch 5600 selesai
Chunk 4 -> Batch 5800 selesai
Chunk 4 -> Batch 6000 selesai
Chunk 4 -> Batch 6200 selesai
Chunk 4 -> Batch 6